### PushT 1000-episode sim with Random Policy

In [3]:
import gymnasium as gym
import gym_pusht
import numpy as np 

env = gym.make("gym_pusht/PushT-v0", render_mode="rgb_array")

objc[32161]: Class AVFFrameReceiver is implemented in both /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x11be683a8) and /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x14c3f43a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[32161]: Class AVFAudioReceiver is implemented in both /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x11be683f8) and /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x14c3f43f8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[32161]: Class SDLApplication is implemented in both /Users/snudurup

## 1. The Random policy

In [ ]:
success_lengths = []   # step counts of episodes that ended in success
failure_lengths = []   # step counts of episodes that hit the 300-step limit

num_ep = 100
for ep in range(num_ep):
    obs, info = env.reset(seed=ep)
    steps = 0
    terminated = False

    while steps < 300 and not terminated:
        action = env.action_space.sample() #random policy
        obs, reward, terminated, truncated, info = env.step(action)
        done = truncated or terminated
        steps += 1
     
    if terminated:
        success_lengths.append(steps)
    else:
        failure_lengths.append(steps)

n_success = len(success_lengths)
mean_success = sum(success_lengths) / n_success if n_success > 0 else 0.0
mean_failure = sum(failure_lengths) / len(failure_lengths) if len(failure_lengths) > 0 else 0.0
print("successes:", n_success, "out of ", num_ep)
print("mean success length (steps):", mean_success)
print("mean failure length (steps):", mean_failure)

objc[24605]: Class SDLApplication is implemented in both /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x116eb8890) and /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x1188252c8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[24605]: Class SDLAppDelegate is implemented in both /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x116eb88e0) and /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x118825318). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[24605]: Class SDLTranslatorResponder is implemented in both /Users/snudurupati/Projects/phy

successes: 0 out of  100
mean success length (steps): 0.0
mean failure length (steps): 300.0


## 2. The Cowboy Policy

In [1]:

def oneline_policy(obs, info):
    goal = np.array([info["goal_pose"][0], info["goal_pose"][1]])
    block = np.array([obs[2], obs[3]])
    margin = 10                                         #margin in pixels, TBD by experimentation
    d = block - goal                                    #points from goal toward block
    norm = np.linalg.norm(d)                            #the length of a vector
    u = d/norm if norm > 0 else np.array([0.0, 0.0])    #same direction, length 1
    action = block + u * margin
    return action


### Same 1000-episode Sim with Cowboy Policy

In [ ]:
import gymnasium as gym
import gym_pusht
import numpy as np 

env = gym.make("gym_pusht/PushT-v0", render_mode="rgb_array")

success_lengths = []   # step counts of episodes that ended in success
failure_lengths = []   # step counts of episodes that hit the 300-step limit

num_ep = 100
for ep in range(num_ep):
    obs, info = env.reset(seed=ep)
    steps = 0
    terminated = False

    while steps < 300 and not terminated:
        action = oneline_policy(obs, info) #defined policy
        obs, reward, terminated, truncated, info = env.step(action)
        done = truncated or terminated
        steps += 1
     
    if terminated:
        success_lengths.append(steps)
    else:
        failure_lengths.append(steps)

n_success = len(success_lengths)
mean_success = sum(success_lengths) / n_success if n_success > 0 else 0.0
mean_failure = sum(failure_lengths) / len(failure_lengths) if len(failure_lengths) > 0 else 0.0
print("successes:", n_success, "out of ", num_ep)
print("mean success length (steps):", mean_success)
print("mean failure length (steps):", mean_failure)

successes: 0 out of  100
mean success length (steps): 0.0
mean failure length (steps): 300.0


## The Cowboy with Human Visibility

In [5]:
env.close()
env = gym.make("gym_pusht/PushT-v0", render_mode="human")

obs, info = env.reset(seed=0)
for steps in range(300):
    action = oneline_policy(obs, info)
    obs, reward, terminated, truncated, info = env.step(action)
    env.render() 
    if steps % 50 == 0:
        print(steps,
              "pusher", np.round(obs[0:2], 1),
              "block", np.round(obs[2:4], 1),
              "action", np.round(action, 1),
              "coverage", round(info["coverage"], 2))
    if terminated:
        break

0 pusher [343.9 295.4] block [241.5 268.5] action [234.  275.1] coverage 0.27
50 pusher [412.1 105.6] block [423.6 181.6] action [431.2 146.2] coverage 0.0
100 pusher [123.6  92.3] block [178.1  37.7] action [145.6  48.2] coverage 0.0
150 pusher [279.7 323.5] block [202.9 319.3] action [232.  334.4] coverage 0.39
200 pusher [338.2  63.8] block [373.7 132.1] action [369.   95.8] coverage 0.0
250 pusher [ 52.  143.1] block [82.4 72.2] action [58.  94.4] coverage 0.0


## 3. Diffusion Policy

```
hf download lerobot/diffusion_pusht README.md config.json train_config.json eval_info.json --local-dir weeks/week-03/model/diffusion_pusht

ls weeks/week-03/modelhf download lerobot/diffusion_pusht model.safetensors --local-dir weeks/week-03/model/diffusion_pusht
```

In [ ]:
import time
import torch
import gymnasium as gym
import gym_pusht
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy

from safetensors.torch import load_file
B = load_file("/Users/snudurupati/Projects/physical-ai-data-architect/weeks/week-03/model/diffusion_pusht/model.safetensors")
MEAN = B["normalize_inputs.buffer_observation_image.mean"].view(3,1,1)
STD  = B["normalize_inputs.buffer_observation_image.std"].view(3,1,1)
S_MIN = B["normalize_inputs.buffer_observation_state.min"]
S_MAX = B["normalize_inputs.buffer_observation_state.max"]
A_MIN = B["unnormalize_outputs.buffer_action.min"]
A_MAX = B["unnormalize_outputs.buffer_action.max"]

def to_batch(obs):
    img = torch.from_numpy(obs["pixels"]).permute(2, 0, 1).float() / 255.0
    img = (img - MEAN) / STD
    st = torch.from_numpy(obs["agent_pos"]).float()
    st = (st - S_MIN) / (S_MAX - S_MIN)
    return {"observation.image": img.unsqueeze(0),   # add batch dimension
            "observation.state": st.unsqueeze(0)}

P = "/Users/snudurupati/Projects/physical-ai-data-architect/weeks/week-03/model/diffusion_pusht"

env = gym.make("gym_pusht/PushT-v0", obs_type="pixels_agent_pos", render_mode="rgb_array")

DEV = "mps"
policy = DiffusionPolicy.from_pretrained(P)
policy.to(DEV); policy.eval(); policy.reset()

for ep in range(5):
    obs, info = env.reset(seed=ep)
    t0 = time.time(); steps = 0; done = False; max_cov = 0.0
    while not done and steps < 300:
        batch = to_batch(obs)                                   # cpu tensors
        batch = {k: v.to(DEV) for k, v in batch.items()}        # whole batch → MPS
        with torch.inference_mode():
            a = policy.select_action(batch)
        a = a.squeeze(0).cpu()                                  # back to CPU first
        action = (a * (A_MAX - A_MIN) + A_MIN).numpy()          # then un-normalize
        obs, r, term, trunc, info = env.step(action)
        max_cov = max(max_cov, info["coverage"])
        done = term or trunc
        steps += 1
    policy.reset()
    print(f"ep {ep}: steps {steps}, terminated {term}, "
          f"max_coverage {max_cov:.3f}, {time.time()-t0:.1f}s")

Loading weights from local directory


ep 0: steps 300, terminated False, max_coverage 0.266, 133.7s
ep 1: steps 300, terminated False, max_coverage 0.000, 181.0s
ep 2: steps 300, terminated False, max_coverage 0.000, 230.9s
ep 3: steps 300, terminated False, max_coverage 0.001, 334.9s
